# exp05b - AR-LRX Rossmann: verifikasi, ablasi gerbang, dan uji skala asli## Mengapa eksperimen ini adaexp05a sudah menunjukkan hasil yang kuat: `AR-LRX [struct_linear]` menang empat arah(vs naif, vs XGBoost polos, vs kerangka lama, vs tahap pertamanya sendiri) pada**ketiga** varian yang sah. Namun ada tiga celah yang **pasti** ditanyakan reviewerQ1/Q2, dan ketiganya ditutup di sini tanpa mengubah satu pun keputusan pemodelan.| Celah pada exp05a | Yang ditutup exp05b ||---|---|| **C1.** Tabel utama melaporkan RMSE **skala asli**, tetapi uji Diebold-Mariano dihitung pada **skala log**. Reviewer berhak menuntut uji signifikansi pada skala yang dilaporkan. | Uji DM dijalankan pada **kedua skala**, disandingkan. || **C2.** Ablasi belum memisahkan sumbangan gerbang **pada tahap pertama yang baik**. Baris `+ gerbang saja` memakai `S1 = linear`; tidak ada sel `S1 struktural tanpa gerbang`. | Ablasi lengkap 3x2 (tiga `S1` x gerbang mati/hidup), dihitung **tanpa melatih ulang apa pun**. || **C3.** Tidak ada bukti bahwa `w` pilihan validation mendekati `w` terbaik di test. Tanpa itu, gerbang bisa dituduh sekadar parameter tambahan yang beruntung. | Kurva sensitivitas RMSE test terhadap `w` penuh, dengan penanda `w` pilihan validation. **Post-hoc, bukan pemilihan.** |## Yang TIDAK berubahProtokol, seed, split, grid, dan definisi model identik dengan exp05a. Bagian 2memverifikasi bahwa seluruh angka exp05a **tereproduksi persis**. Bila verifikasi itululus, exp05b menggantikan exp05a sebagai sumber tabel naskah; bila gagal, adamasalah lingkungan yang harus diselesaikan lebih dulu.## PrasyaratGanti `src/experiments/arlrx.py` dengan versi terbaru (yang menyimpan`_test_correction`), lalu **Restart Kernel**. Penambahan itu murni penyimpananprediksi; tidak ada satu pun angka yang berubah karenanya.

In [ ]:
import sys, os, json, warningssys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))warnings.filterwarnings("ignore")import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom src.experiments import protocol as Pfrom src.experiments import arlrx as AP.set_global_seed()EXPERIMENT = "exp05b_rossmann_arlrx_audit"pd.set_option("display.width", 250)pd.set_option("display.max_columns", 60)print("Lingkungan:", P.environment_stamp())# Prasyarat: arlrx.py versi baruimport inspectassert "_test_correction" in inspect.getsource(A.run_arlrx), (    "arlrx.py masih versi lama. Ganti berkasnya, lalu Restart Kernel.")print("arlrx.py: versi dengan penyimpanan koreksi  OK")

## 1. Data dan model - identik dengan exp05aSembilan model per varian, urutan dan argumen sama persis. Satu-satunya perbedaan:seluruh prediksi test disimpan ke cakram, sehingga setiap analisis lanjutan(residual per toko, dekomposisi galat, kurva gerbang, uji apa pun) dapat dilakukandi kemudian hari **tanpa melatih ulang**.

In [ ]:
frame = P.build_rossmann_frame("../data/raw/rossmann/train.csv",                               "../data/raw/rossmann/store.csv")VARIANTS = ["V1_customers_dropped", "V2_customers_lagged", "V3_sales_lagged"]QUICK_RUN = False   # biarkan False: exp05b harus sebanding dengan exp05aif QUICK_RUN:    keep = np.sort(frame["Store"].unique())[:80]    frame = frame[frame["Store"].isin(keep)].reset_index(drop=True)    XGB_GRID = {"n_estimators": [300], "max_depth": [6], "learning_rate": [0.1],                "subsample": [0.8], "colsample_bytree": [0.8], "max_bin": [256]}else:    XGB_GRID = A.GRID_XGB_ARLRXdatasets = {v: P.build_rossmann_dataset(frame, v, target="log1p") for v in VARIANTS}print("Bentuk kerangka:", frame.shape, "| toko:", frame["Store"].nunique())

In [ ]:
rows = []for variant in VARIANTS:    d = datasets[variant]    rows.append(P.rossmann_seasonal_naive(d, inverse_transform=np.expm1))    for kind in A.STAGE1_KINDS:        rows.append(A.run_stage1_only(f"S1 [{kind}]", d, kind,                                      inverse_transform=np.expm1))        rows.append(A.run_arlrx(f"AR-LRX [{kind}]", d, kind, XGB_GRID,                                inverse_transform=np.expm1))        print(f"  {variant:24s} S1={kind:14s} selesai", flush=True)    rows.append(P.run_model("XGBoost", P.fp_xgboost, d, XGB_GRID,                            inverse_transform=np.expm1))    rows.append(P.run_model("LR-XGB (residual, tanpa gerbang)", P.fp_lr_xgb_residual,                            d, XGB_GRID, inverse_transform=np.expm1))    print(f"  {variant:24s} pembanding selesai", flush=True)results = P.save_results(rows, EXPERIMENT)by_key = {(r["feature_set"], r["model"]): r for r in rows}print(f"\n{len(results)} baris ditulis ke ../results/{EXPERIMENT}.csv")

In [ ]:
# Simpan seluruh prediksi test agar analisis lanjutan tidak perlu melatih ulang.store = {}for (variant, model), r in by_key.items():    tag = f"{variant}|{model}"    store[f"{tag}|test_pred"] = np.asarray(r["_test_pred"], dtype=np.float32)    for extra_key in ("_stage1_test_raw", "_test_correction"):        if extra_key in r:            store[f"{tag}|{extra_key[1:]}"] = np.asarray(r[extra_key], dtype=np.float32)for v in VARIANTS:    store[f"{v}|y_test"] = np.asarray(datasets[v].y_test, dtype=np.float32)path = f"../results/{EXPERIMENT}_predictions.npz"np.savez_compressed(path, **store)print(f"{len(store)} larik disimpan ke {path} "      f"({os.path.getsize(path)/1e6:.1f} MB)")

## 2. Verifikasi reproduksi terhadap exp05aAngka exp05b harus identik dengan exp05a sampai batas presisi mesin. Bila ada satusaja baris yang meleset, seluruh analisis di bawah tidak boleh dipakai.

In [ ]:
ref_path = "../results/exp05a_rossmann_arlrx.csv"if os.path.exists(ref_path):    ref = pd.read_csv(ref_path)    key = ["feature_set", "model"]    cmp_cols = ["test_RMSE", "orig_RMSE", "orig_MAE", "orig_R2", "gate_w"]    merged = (results[key + cmp_cols]              .merge(ref[key + cmp_cols], on=key, suffixes=("_b", "_a")))    bad = []    for c in cmp_cols:        diff = (merged[f"{c}_b"] - merged[f"{c}_a"]).abs()        tol = 1e-9 if c == "gate_w" else 1e-6        n_bad = int((diff > tol).sum())        print(f"  {c:12s}: maks selisih = {diff.max():.3e}   melebihi toleransi: {n_bad}")        if n_bad:            bad.append(c)    print("\nREPRODUKSI PERSIS:", not bad)    if bad:        print("  Kolom bermasalah:", bad)        display(merged[key + [f"{c}_a" for c in bad] + [f"{c}_b" for c in bad]])else:    print(f"{ref_path} tidak ditemukan - lewati verifikasi.")

## 3. Uji Diebold-Mariano pada DUA skalaNaskah melaporkan RMSE skala asli, maka signifikansi juga harus diuji di sana.Skala log tetap dilaporkan agar dapat dibandingkan dengan exp05a dan denganliteratur yang bekerja pada target tertransformasi.Nilai DM negatif berarti AR-LRX lebih akurat. Kolom `sepakat` menandai apakah keduaskala memberi kesimpulan yang sama - inilah yang sebenarnya ingin dilihat reviewer.

In [ ]:
dm_rows = []for variant in VARIANTS:    d = datasets[variant]    y_log = d.y_test    y_orig = np.expm1(y_log)    for kind in A.STAGE1_KINDS:        prop = by_key[(variant, f"AR-LRX [{kind}]")]        refs = [("XGBoost polos", by_key[(variant, "XGBoost")]),                ("Naif per toko", by_key[(variant, "SeasonalNaive(store x dow x promo median)")]),                ("Kerangka lama", by_key[(variant, "LR-XGB (residual, tanpa gerbang)")]),                (f"S1 [{kind}] sendirian", by_key[(variant, f"S1 [{kind}]")])]        for label, ref in refs:            t_log = P.diebold_mariano(y_log, prop["_test_pred"], ref["_test_pred"])            t_org = P.diebold_mariano(y_orig, np.expm1(prop["_test_pred"]),                                      np.expm1(ref["_test_pred"]))            win_log = t_log["DM"] < 0            win_org = t_org["DM"] < 0            sig_log = t_log["p_value"] < 0.05            sig_org = t_org["p_value"] < 0.05            dm_rows.append({                "varian": variant, "S1": kind, "pembanding": label,                "DM (log)": round(t_log["DM"], 3), "p (log)": t_log["p_value"],                "DM (asli)": round(t_org["DM"], 3), "p (asli)": t_org["p_value"],                "menang log": bool(win_log) and bool(sig_log),                "menang asli": bool(win_org) and bool(sig_org),                "sepakat": (bool(win_log) == bool(win_org)),            })dm2 = pd.DataFrame(dm_rows)dm2.to_csv(f"../results/{EXPERIMENT}_dm_two_scales.csv", index=False)display(dm2.to_string(index=False))print(f"\nKedua skala sepakat arahnya : {int(dm2.sepakat.sum())} dari {len(dm2)}")print(f"Menang signifikan (log)     : {int(dm2['menang log'].sum())} dari {len(dm2)}")print(f"Menang signifikan (asli)    : {int(dm2['menang asli'].sum())} dari {len(dm2)}")print("\nKonfigurasi utama saja (S1 = struct_linear):")display(dm2[dm2.S1 == "struct_linear"].to_string(index=False))

## 4. Ablasi lengkap: memisahkan tahap pertama dari gerbangAblasi yang benar mematikan **satu** komponen sambil menahan sisanya tetap. Karena`w` tidak memengaruhi pelatihan tahap kedua, "gerbang dimatikan" (`w = 1`, koreksipenuh seperti kerangka lama) dapat dihitung dari model yang **sama persis** - tanpamelatih ulang, dan tanpa pemilihan hyperparameter yang berbeda. Ini justru ablasiyang lebih bersih daripada melatih ulang dengan `w` dipatok, karena satu-satunyayang berubah benar-benar hanya gerbang.Sel `w = 0` adalah tahap pertama sendirian; `w = w*` adalah AR-LRX.

In [ ]:
def metrics_at_w(r, y_log, w, clip=True):    """Metrik skala asli untuk gerbang sembarang, dari prediksi tersimpan."""    pred = r["_stage1_test_raw"] + w * r["_test_correction"]    if clip:        pred = np.maximum(pred, 0.0)    return (P.compute_metrics(np.expm1(y_log), np.expm1(pred), prefix="orig_"),            float(np.sqrt(np.mean((y_log - pred) ** 2))))abl2 = []for variant in VARIANTS:    d = datasets[variant]    old = by_key[(variant, "LR-XGB (residual, tanpa gerbang)")]["orig_RMSE"]    for kind in A.STAGE1_KINDS:        r = by_key[(variant, f"AR-LRX [{kind}]")]        m0, _ = metrics_at_w(r, d.y_test, 0.0)   # tahap pertama sendirian        m1, _ = metrics_at_w(r, d.y_test, 1.0)   # gerbang dimatikan (koreksi penuh)        mw, _ = metrics_at_w(r, d.y_test, r["gate_w"])        abl2.append({            "varian": variant, "S1": kind, "w*": r["gate_w"],            "w=0 (S1 saja)": m0["orig_RMSE"],            "w=1 (gerbang mati)": m1["orig_RMSE"],            "w=w* (AR-LRX)": mw["orig_RMSE"],            "sumbangan gerbang (%)": (mw["orig_RMSE"] - m1["orig_RMSE"]) / m1["orig_RMSE"] * 100,            "sumbangan koreksi (%)": (mw["orig_RMSE"] - m0["orig_RMSE"]) / m0["orig_RMSE"] * 100,            "vs kerangka lama (%)": (mw["orig_RMSE"] - old) / old * 100,        })abl2 = pd.DataFrame(abl2)abl2.to_csv(f"../results/{EXPERIMENT}_ablation_full.csv", index=False)print("Negatif = lebih baik. 'sumbangan gerbang' menahan segalanya tetap kecuali w.")display(abl2.round(3).to_string(index=False))print("\nRingkasan sumbangan gerbang per jenis tahap pertama (rata-rata 3 varian):")display(abl2.groupby("S1")[["sumbangan gerbang (%)", "sumbangan koreksi (%)"]]        .mean().round(3))

## 5. Kurva sensitivitas gerbang**Peringatan metodologis:** kurva ini dihitung SETELAH `w` ditetapkan darivalidation. Ia dipakai untuk *mendiagnosis*, bukan untuk memilih. Titik penandamenunjukkan `w` pilihan validation; garis putus-putus menunjukkan `w` terbaik ditest yang **tidak pernah dipakai**. Jarak antara keduanya adalah ukuran seberapabaik validation memandu pemilihan gerbang.

In [ ]:
curve = []for variant in VARIANTS:    d = datasets[variant]    for kind in A.STAGE1_KINDS:        r = by_key[(variant, f"AR-LRX [{kind}]")]        for w in np.round(np.linspace(0, 1, 21), 3):            m, rmse_log = metrics_at_w(r, d.y_test, float(w))            curve.append({"varian": variant, "S1": kind, "w": float(w),                          "RMSE asli": m["orig_RMSE"], "RMSE log": rmse_log})curve = pd.DataFrame(curve)curve.to_csv(f"../results/{EXPERIMENT}_gate_curve.csv", index=False)gap = []for (variant, kind), g in curve.groupby(["varian", "S1"]):    r = by_key[(variant, f"AR-LRX [{kind}]")]    w_star = r["gate_w"]    w_test = float(g.loc[g["RMSE asli"].idxmin(), "w"])    rmse_star = float(g.loc[np.isclose(g.w, w_star), "RMSE asli"].iloc[0])    rmse_best = float(g["RMSE asli"].min())    gap.append({"varian": variant, "S1": kind,                "w* (validation)": w_star, "w terbaik di test": w_test,                "RMSE di w*": rmse_star, "RMSE di w terbaik": rmse_best,                "harga kesalahan pilih (%)": (rmse_star - rmse_best) / rmse_best * 100})gap = pd.DataFrame(gap)print("Seberapa mahal memilih w dari validation, bukan dari test:")display(gap.round(4).to_string(index=False))print(f"\nRata-rata harga kesalahan pilih: {gap['harga kesalahan pilih (%)'].mean():.4f}%")print(f"Maksimum                        : {gap['harga kesalahan pilih (%)'].max():.4f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), dpi=140, sharex=True)for ax, variant in zip(axes, VARIANTS):    for kind in A.STAGE1_KINDS:        g = curve[(curve.varian == variant) & (curve.S1 == kind)]        ax.plot(g.w, g["RMSE asli"], marker="o", ms=3, lw=1.4, label=f"S1 = {kind}")        r = by_key[(variant, f"AR-LRX [{kind}]")]        ws = r["gate_w"]        ys = float(g.loc[np.isclose(g.w, ws), "RMSE asli"].iloc[0])        ax.scatter([ws], [ys], s=150, marker="*", zorder=5, edgecolor="k", linewidth=.6)    ax.set_title(variant, fontsize=10)    ax.set_xlabel("gerbang w")    ax.grid(alpha=.25)axes[0].set_ylabel("RMSE skala asli")axes[0].legend(fontsize=8)fig.suptitle("Sensitivitas RMSE test terhadap gerbang  (bintang = w pilihan validation)",             fontsize=11)plt.tight_layout(); plt.show()

## 6. Di mana AR-LRX menang atas XGBoost?Pemecahan galat menurut segmen. Ini bukan sekadar pelengkap: ia menjelaskan*mengapa* kerangka ini bekerja, dan itulah yang membedakan sumbangan metodologisdari sekadar angka yang lebih kecil.

In [ ]:
seg_rows = []for variant in VARIANTS:    d = datasets[variant]    y = np.expm1(d.y_test)    a = np.expm1(by_key[(variant, "AR-LRX [struct_linear]")]["_test_pred"])    x = np.expm1(by_key[(variant, "XGBoost")]["_test_pred"])    fn = list(d.feature_names)    Xt = d.X_test    promo = Xt[:, fn.index("Promo")] if "Promo" in fn else np.zeros(len(y))    dow = Xt[:, fn.index("DayOfWeek")] if "DayOfWeek" in fn else np.zeros(len(y))    def rmse(t, p, mask):        return float(np.sqrt(np.mean((t[mask] - p[mask]) ** 2))) if mask.sum() else np.nan    segments = {"semua": np.ones(len(y), bool),                "promo": promo == 1, "tanpa promo": promo == 0}    for dv in sorted(set(np.unique(dow).tolist()))[:7]:        segments[f"hari {int(dv)}"] = dow == dv    # desil volume toko berdasarkan rata-rata target di TEST hanya untuk pengelompokan    # deskriptif (tidak dipakai untuk pemodelan apa pun)    if "Store" in fn:        st = Xt[:, fn.index("Store")]        mean_by_store = pd.Series(y).groupby(pd.Series(st)).transform("mean").to_numpy()        q = pd.Categorical(pd.qcut(mean_by_store, 4,                                   labels=["Q1 kecil", "Q2", "Q3", "Q4 besar"],                                   duplicates="drop"))        for lab in q.categories:            segments[f"toko {lab}"] = np.asarray(q == lab)    for name, mask in segments.items():        ra, rx = rmse(y, a, mask), rmse(y, x, mask)        seg_rows.append({"varian": variant, "segmen": name, "n": int(mask.sum()),                         "AR-LRX": ra, "XGBoost": rx,                         "selisih (%)": (ra - rx) / rx * 100})seg = pd.DataFrame(seg_rows)seg.to_csv(f"../results/{EXPERIMENT}_segments.csv", index=False)print("Negatif = AR-LRX lebih baik.")for v in VARIANTS:    print(f"\n--- {v} ---")    display(seg[seg.varian == v].round(3).to_string(index=False))

## 7. Ringkasan untuk naskahSel di bawah mencetak kalimat-kalimat yang dapat langsung dipakai, dengan angkayang terisi dari hasil sebenarnya - supaya tidak ada angka yang disalin tangan.

In [ ]:
print("="*88)for variant in VARIANTS:    sub = results[results.feature_set == variant].set_index("model")    b = sub.loc["AR-LRX [struct_linear]"]    nv = sub.loc["SeasonalNaive(store x dow x promo median)", "orig_RMSE"]    xg = sub.loc["XGBoost", "orig_RMSE"]    ol = sub.loc["LR-XGB (residual, tanpa gerbang)", "orig_RMSE"]    s1 = sub.loc["S1 [struct_linear]", "orig_RMSE"]    dsub = dm2[(dm2.varian == variant) & (dm2.S1 == "struct_linear")].set_index("pembanding")    print(f"\n{variant}")    print(f"  AR-LRX [struct_linear], w* = {b.gate_w}, RMSE = {b.orig_RMSE:.2f} "          f"(MAE {b.orig_MAE:.2f}, RMSPE {b.orig_RMSPE:.5f}, R2 {b.orig_R2:.4f})")    for lab, ref, key in [("naif per toko", nv, "Naif per toko"),                          ("XGBoost polos", xg, "XGBoost polos"),                          ("kerangka lama", ol, "Kerangka lama"),                          ("tahap pertama sendirian", s1, "S1 [struct_linear] sendirian")]:        d_ = (b.orig_RMSE - ref) / ref * 100        p_ = dsub.loc[key, "p (asli)"]        print(f"    lebih baik {abs(d_):5.2f}% dari {lab:24s} (DM skala asli, p = {p_:.3g})")print("\n" + "="*88)print(f"Kedua skala DM sepakat pada {int(dm2.sepakat.sum())}/{len(dm2)} perbandingan.")print(f"Harga memilih w dari validation, bukan test: rata-rata "      f"{gap['harga kesalahan pilih (%)'].mean():.4f}%, maksimum "      f"{gap['harga kesalahan pilih (%)'].max():.4f}%.")print("Berkas yang dihasilkan:")for f in sorted(os.listdir("../results")):    if f.startswith(EXPERIMENT):        print(f"  ../results/{f}")